# Causal methods: ATE estimation and uplift

- Simulate an observational A/B experiment (discount vs no discount) with confounding.
- Estimate ATE using: **naive difference-in-means**, **regression adjustment**, **Difference-in-Differences**, **Propensity Score Matching**.
- Compare estimates to **known ground truth** and report bias; use **bootstrap CIs** where applicable.
- **Uplift (T-learner)** for heterogeneous effects and **targeted discount allocation**.

In [3]:
# Allow notebook to import from src/ folder
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.simulate_data import SimConfig, simulate_discount_data, to_user_level_post
from src.did import did_ate
from src.psm import psm_ate
from src.regression_ate import regression_ate
from src.uplift import t_learner_uplift, targeting_simulation
from src.utils import diff_in_means_ate

# Simulate observational A/B experiment (confounded treatment assignment)
cfg = SimConfig(n_users=8000, seed=42)
df, meta = simulate_discount_data(cfg)
df_post = to_user_level_post(df)
ate_true = meta["ate_true_post"]

# --- ATE estimators ---
naive_ate = diff_in_means_ate(df_post, "Y", "T")
reg_result = regression_ate(df_post, n_boot=500)
did_result = did_ate(df)
psm_result = psm_ate(df_post, caliper=0.05)

print("Ground truth ATE (post):", round(ate_true, 4))
print("Naive (diff-in-means):", round(naive_ate, 4))
print("Regression adjustment:", round(reg_result["regression_ate"], 4), "CI:", reg_result["bootstrap_ci"])
print("DiD:", round(did_result["did_effect"], 4), "SE:", round(did_result["std_err"], 4))
print("PSM:", round(psm_result["psm_ate"], 4), "CI:", psm_result["bootstrap_ci"])

# --- Bias vs ground truth ---
results = [
    ("Naive", naive_ate, None),
    ("Regression adj.", reg_result["regression_ate"], reg_result["bootstrap_ci"]),
    ("DiD", did_result["did_effect"], {"ci_low": did_result["ci_low"], "ci_high": did_result["ci_high"]}),
    ("PSM", psm_result["psm_ate"], psm_result["bootstrap_ci"]),
]
bias_table = pd.DataFrame([
    {"Estimator": name, "ATE": est, "Bias": est - ate_true, "Bias %": (est - ate_true) / (ate_true + 1e-9) * 100}
    for name, est, _ in results
])
print("\nBias relative to ground truth:")
print(bias_table.to_string(index=False))

# --- Uplift (T-learner) and targeted discount allocation ---
scored = t_learner_uplift(df_post)
targeting = targeting_simulation(scored, top_frac=0.3)
print("\nTargeted discount (top 30% by uplift) vs blanket vs no discount:")
print(targeting)

ImportError: cannot import name 'SimConfig' from 'src.simulate_data' (/Users/potta/Desktop/causal-discount-project/src/simulate_data.py)